**Table of contents**<a id='toc0_'></a>    
- [1. LSTM: Why it is needed and how the four gates work](#toc1_1_)    
    - [Step 1: Dicide what to forget (Forget Gate)](#toc1_1_1_)    
    - [Step 2: Decide what new information to Add (Input gate + Candidate memory)](#toc1_1_2_)    
    - [Step 3: Update the Cell State (Memory Highway)](#toc1_1_3_)    
    - [Step 4: Generate the Output (Output Gate)](#toc1_1_4_)    
    - [LSTM cell architecture](#toc1_1_5_)    
- [2. RNN basics: LSTM cell, Encoder, Decoder, Seq2Seq baseline](#toc2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[1. LSTM: Why it is needed and how the four gates work](#toc0_)

A vanilla RNN can suffer from *vanishing gradients* on long sequences: the gradient becomes smaller across time steps, making it difficult to preserve information from the beginning of a sentence.

LSTM adds a separate memory path called the **cell state** $c_t$. This state is updated through an additive path, which helps gradients remain more stable. At each time step, an LSTM uses four components:

- **Forget gate** $f_t$: decides how much information to keep from $c_{t-1}$; sigmoid output is in $(0, 1)$.
- **Input gate** $i_t$: decides how much new information to write into the cell state; uses sigmoid.
- **Candidate state** $\tilde{c}_t$: new content that may be written into memory; uses $\tanh$, whose output is in $(-1, 1)$.
- **Output gate** $o_t$: decides how much information from the cell state becomes the hidden state; uses sigmoid.

For current input $x_t$, previous hidden state $h_{t-1}$, and previous cell state $c_{t-1}$:

$$
\begin{aligned}
i_t &= \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i), \\
f_t &= \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f), \\
\tilde{c}_t &= \tanh(x_t W_{xc} + h_{t-1} W_{hc} + b_c), \\
o_t &= \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o).
\end{aligned}
$$

The memory and hidden state are then updated as follows:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

The addition in the $c_t$ equation creates a direct memory path. This is the key mechanism that helps LSTM reduce the vanishing-gradient problem compared with a vanilla RNN.

### <a id='toc1_1_1_'></a>[Step 1: Dicide what to forget (Forget Gate)](#toc0_)

The forget gate takes the previous hidden state $h_{t-1}$ and the current input $x_{t}$.

It produces values between 0 and 1 using a sigmoid.

These values determine how much of the previous cell state $c_{t - 1}$ should be kept.
 
Values near 1 mean the information is reetained.

Values near 0 mean the information is discarded.

This mechanism helps remove outdated or irrelevant context from long sequences.

### <a id='toc1_1_2_'></a>[Step 2: Decide what new information to Add (Input gate + Candidate memory)](#toc0_)

The input gate determines how much of the new information should be written into the memory.

The candidate memory creates a new vector of potential information using a Tanh.

The input gate filters the candidate memory before it is added to the cell state.

This ensures that only relevant new information updates the internal memory.

### <a id='toc1_1_3_'></a>[Step 3: Update the Cell State (Memory Highway)](#toc0_)

The updated cell state is formed by combining the scaled previous memory and the filtered new memory.

The forget gate reduces or removes parts of the old memory.

The input gate and candidate memory add new information to fill in or modify the cell state.

Additive updates help gradients flow more consistently, preventing the vanishing gradient problem.

This stable memory path enables learning complex long-range patterns.

### <a id='toc1_1_4_'></a>[Step 4: Generate the Output (Output Gate)](#toc0_)

The output gate determines how much of the updated cell state influences the hidden state $h_{t}$.

A sigmoid activation computes this output filter.

The filtered value is merged with the tanh-transformed cell state to produce the final hidden state.

The hidden state serves as the LSTM’s output at time step t.

The same hidden state is passed forward to the next time step for continued processing.



### <a id='toc1_1_5_'></a>[LSTM cell architecture](#toc0_)

![LSTM cell architecture](../../images/architecture.png)

+ Yellow nodes represent element-wise multiplication ($\odot$); 
+ Green node represents element-wise addition.

$$
\begin{aligned}
i_t &= \sigma(x_t W_{xi} + h_{t-1} W_{hi} + b_i), \\
f_t &= \sigma(x_t W_{xf} + h_{t-1} W_{hf} + b_f), \\
\tilde{c}_t &= \tanh(x_t W_{xc} + h_{t-1} W_{hc} + b_c), \\
o_t &= \sigma(x_t W_{xo} + h_{t-1} W_{ho} + b_o).
\end{aligned}
$$

https://codingowen.github.io/projects/lstm_from_scratch/

# <a id='toc2_'></a>[2. RNN basics: LSTM cell, Encoder, Decoder, Seq2Seq baseline](#toc0_)

xây baseline Seq2Seq LSTM **KHÔNG có Attention**


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


import pickle
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from src.vocab import Vocab

torch.manual_seed(42)   # cố định random để kết quả LẶP LẠI ĐƯỢC mỗi lần chạy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Running on:", device)

Running on: cuda


In [3]:
DATA_DIR = 'D:/AIResearcher/MachineTraslation/data/processed'

with open(f'{DATA_DIR}/en_vocab.pkl', 'rb') as f:
    en_vocab = pickle.load(f)
with open(f'{DATA_DIR}/vi_vocab.pkl', 'rb') as f:
    vi_vocab = pickle.load(f)
with open(f'{DATA_DIR}/train_ids.pkl', 'rb') as f:
    train_ids = pickle.load(f)
with open(f'{DATA_DIR}/dev_ids.pkl', 'rb') as f:
    dev_ids = pickle.load(f)
with open(f'{DATA_DIR}/test_ids.pkl', 'rb') as f:
    test_ids = pickle.load(f)

In [4]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs # list[(src_ids: list[int], tgt_ids: list[int])]
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

In [5]:
train_dataset = TranslationDataset(train_ids) # list[(src_ids: list[int], tgt_ids: list[int])]
dev_dataset   = TranslationDataset(dev_ids)
test_dataset  = TranslationDataset(test_ids)

In [6]:
PAD_IDX = 0 # đã định nghĩa trong Vocab: {'<pad>': 0, ...}
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lengths = torch.tensor([len(s) for s in src_batch])
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    # tgt_padded: [<bos>, i, am, a, student, <eos>]
    # tgt_input  = [<bos>, i, am, a, student] bỏ token CUỐI
    # tgt_output = [i, am, a, student, <eos>] bỏ token ĐẦU
    tgt_input  = tgt_padded[:, :-1]   # INPUT cho decoder
    tgt_output = tgt_padded[:, 1:]    # TARGET để tính loss
    return src_padded, src_lengths, tgt_input, tgt_output


In [7]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(type(train_loader))

<class 'torch.utils.data.dataloader.DataLoader'>


In [8]:
embed_dim = 3
hidden_dim = 2

In [9]:
x_t = torch.tensor([[0.5, -0.2, 0.1]])
h_prev = torch.tensor([[0.1, 0.2]])
c_prev = torch.tensor([[0.0, 0.0]])

W_xi, W_hi = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xf, W_hf = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xc, W_hc = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xo, W_ho = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))

print(W_xi)
print(W_xf)
print(W_xc)
print(W_xo)

print()

print(f"x_t @ W_xi = {x_t @ W_xi}")
print(f"x_t @ W_xf = {x_t @ W_xf}")
print(f"x_t @ W_xc = {x_t @ W_xc}")
print(f"x_t @ W_xo = {x_t @ W_xc}")

print()

print(f"h_prev @ W_hi = {h_prev @ W_hi}")
print(f"h_prev @ W_hf = {h_prev @ W_hf}")
print(f"h_prev @ W_hc = {h_prev @ W_hc}")
print(f"h_prev @ W_ho = {h_prev @ W_ho}")

print()

# W_x = torch.cat([W_xi, W_xf, W_xc, W_xo], dim=1)
# W_h = torch.cat([W_hi, W_hf, W_hc, W_ho], dim=1) 
W_x = torch.cat([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0)
W_h = torch.cat([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0) 

print(f"Shape W_x = {W_x.shape}")

print(W_x.T)
print()
print(W_h.T)
print()
# print(x_t @ W_x)
# print(h_prev @ W_h)
print(x_t @ W_x.T)
print(h_prev @ W_h.T)


tensor([[0.8823, 0.9150],
        [0.3829, 0.9593],
        [0.3904, 0.6009]])
tensor([[0.9346, 0.5936],
        [0.8694, 0.5677],
        [0.7411, 0.4294]])
tensor([[0.2696, 0.4414],
        [0.2969, 0.8317],
        [0.1053, 0.2695]])
tensor([[0.9516, 0.0753],
        [0.8860, 0.5832],
        [0.3376, 0.8090]])

x_t @ W_xi = tensor([[0.4036, 0.3257]])
x_t @ W_xf = tensor([[0.3675, 0.2262]])
x_t @ W_xc = tensor([[0.0860, 0.0813]])
x_t @ W_xo = tensor([[0.0860, 0.0813]])

h_prev @ W_hi = tensor([[0.2138, 0.1060]])
h_prev @ W_hf = tensor([[0.1419, 0.1829]])
h_prev @ W_hc = tensor([[0.1453, 0.0212]])
h_prev @ W_ho = tensor([[0.1687, 0.1589]])

Shape W_x = torch.Size([8, 3])
tensor([[0.8823, 0.9150, 0.9346, 0.5936, 0.2696, 0.4414, 0.9516, 0.0753],
        [0.3829, 0.9593, 0.8694, 0.5677, 0.2969, 0.8317, 0.8860, 0.5832],
        [0.3904, 0.6009, 0.7411, 0.4294, 0.1053, 0.2695, 0.3376, 0.8090]])

tensor([[0.2566, 0.7936, 0.8854, 0.5739, 0.3588, 0.1994, 0.5779, 0.9040],
        [0.9408, 0.1

In [10]:
lstm_cell = nn.LSTMCell(embed_dim, hidden_dim)
print(lstm_cell.weight_ih.shape)   # torch.Size([8, 3])
print(lstm_cell.weight_hh.shape)   # torch.Size([8, 2])
print(lstm_cell.bias_ih.shape)     # torch.Size([8])
print(lstm_cell.bias_hh.shape)     # torch.Size([8])

print(x_t @ lstm_cell.weight_ih.T + lstm_cell.bias_ih) 
print(h_prev @ lstm_cell.weight_hh.T + lstm_cell.bias_hh) 

torch.Size([8, 3])
torch.Size([8, 2])
torch.Size([8])
torch.Size([8])
tensor([[-0.2021, -0.2790,  0.5899, -0.3886,  0.0623, -0.0673,  0.7861, -0.8587]],
       grad_fn=<AddBackward0>)
tensor([[-0.3646,  0.1904,  0.0230, -0.0696,  0.3728,  0.2999, -0.5031, -0.2347]],
       grad_fn=<AddBackward0>)


In [11]:
embed_dim = 3
hidden_dim = 2

# Input tại bước t, và trạng thái từ bước t-1
# (khởi tạo = 0 vì đây là bước ĐẦU TIÊN của 1 câu)
x_t    = torch.tensor([[0.5, -0.2, 0.1]])   # (batch=1, embed_dim=3)
h_prev = torch.tensor([[0.0, 0.0]])          # (batch=1, hidden_dim=2)  — h_{t-1}
c_prev = torch.tensor([[0.0, 0.0]])          # (batch=1, hidden_dim=2)  — c_{t-1}

# Đặt tên: subscript ĐẦU = nguồn tín hiệu (x hoặc h), subscript SAU = cổng đích (i/f/g/o)
W_xi, W_hi = torch.randn(embed_dim, hidden_dim)*0.5, torch.randn(hidden_dim, hidden_dim)*0.5
W_xf, W_hf = torch.randn(embed_dim, hidden_dim)*0.5, torch.randn(hidden_dim, hidden_dim)*0.5
W_xc, W_hc = torch.randn(embed_dim, hidden_dim)*0.5, torch.randn(hidden_dim, hidden_dim)*0.5
W_xo, W_ho = torch.randn(embed_dim, hidden_dim)*0.5, torch.randn(hidden_dim, hidden_dim)*0.5

# Bias — MỖI CỔNG 1 bias riêng, shape (hidden_dim,)
b_i = torch.randn(hidden_dim)*0.1
b_f = torch.randn(hidden_dim)*0.1
b_c = torch.randn(hidden_dim)*0.1
b_o = torch.randn(hidden_dim)*0.1



In [12]:
# (batch, hidden_dim) + (batch, hidden_dim)
i_t = torch.sigmoid(x_t @ W_xi + h_prev @ W_hi + b_i)   # input gate 
f_t = torch.sigmoid(x_t @ W_xf + h_prev @ W_hf + b_f)   # forget gate
g_t = torch.tanh(   x_t @ W_xc+ h_prev @ W_hc + b_c)   # candidate (= c-tilde_t)
o_t = torch.sigmoid(x_t @ W_xo + h_prev @ W_ho + b_o)   # output gate

c_t = f_t * c_prev + i_t * g_t     # phép * là element-wise (odot), KHÔNG phải nhân ma trận
h_t = o_t * torch.tanh(c_t)

print("i_t:", i_t)
print("f_t:", f_t)
print("g_t:", g_t)
print("o_t:", o_t)
print("c_t:", c_t)
print("h_t:", h_t)

i_t: tensor([[0.4895, 0.4673]])
f_t: tensor([[0.4604, 0.4619]])
g_t: tensor([[0.2737, 0.0453]])
o_t: tensor([[0.5697, 0.5483]])
c_t: tensor([[0.1340, 0.0212]])
h_t: tensor([[0.0759, 0.0116]])


In [13]:
lstm_cell = nn.LSTMCell(embed_dim, hidden_dim)
with torch.no_grad(): # [i, f, c, o]
    lstm_cell.weight_ih.copy_(torch.cat([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0))
    lstm_cell.weight_hh.copy_(torch.cat([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0))
    lstm_cell.bias_ih.copy_(torch.cat([b_i, b_f, b_c, b_o], dim=0))
    lstm_cell.bias_hh.zero_()   

h_t_pytorch, c_t_pytorch = lstm_cell(x_t, (h_prev, c_prev))


print(h_t_pytorch.shape)
print(c_t_pytorch.shape)

print("h_t (tính)        :", h_t)
print("h_t (PyTorch built-in):", h_t_pytorch)
assert torch.allclose(h_t, h_t_pytorch, atol=1e-6), "Not match"

torch.Size([1, 2])
torch.Size([1, 2])
h_t (tính)        : tensor([[0.0759, 0.0116]])
h_t (PyTorch built-in): tensor([[0.0759, 0.0116]], grad_fn=<MulBackward0>)


In [14]:
import torch
A = torch.tensor(1., requires_grad=True)
print("Tensor-A:", A)

with torch.no_grad():
    B = A + 1

print("B:-", B)
print("B.requires_grad=", B.requires_grad)

Tensor-A: tensor(1., requires_grad=True)
B:- tensor(2.)
B.requires_grad= False


In [15]:
import torch
import torch.nn as nn

layer = nn.Linear(1, 4)
X = torch.rand(4, 1)

Y = layer(X)

print(X)

print("X shape:", X.shape)
print("Weight shape:", layer.weight.shape)
print("Bias shape:", layer.bias.shape)
print("Y shape:", Y.shape)

print(Y)

tensor([[0.7797],
        [0.1519],
        [0.7513],
        [0.7269]])
X shape: torch.Size([4, 1])
Weight shape: torch.Size([4, 1])
Bias shape: torch.Size([4])
Y shape: torch.Size([4, 4])
tensor([[-0.4082, -0.0034, -0.5561, -1.5240],
        [-0.2216, -0.3910, -0.3547, -1.0634],
        [-0.3997, -0.0209, -0.5470, -1.5031],
        [-0.3925, -0.0360, -0.5391, -1.4852]], grad_fn=<AddmmBackward0>)
